# Docs 6 — Trust but Verify

The gate before the table: four check families, quarantine with reasons, a
measured error rate, and a spot-check of rows that passed.

In [ ]:
# Twelve extracted rows - the month's merge of rules- and LLM-extracted
# output. Four of them carry seeded errors. Your gate should find all four.
rows = [
 {"source":"jan","date":"2026-01-16","families":167,"donations":1210.00,"approx":False},
 {"source":"feb","date":"2026-02-13","families":174,"donations":1385.50,"approx":False},
 {"source":"mar","date":"2026-03-14","families":212,"donations":1847.50,"approx":False},
 {"source":"apr","date":"2026-04-11","families":198,"donations":2210.00,"approx":False},
 {"source":"may","date":"2026-05-09","families":241,"donations":"l655.25","approx":False},
 {"source":"jun","date":None,"families":188,"donations":1500.00,"approx":True},
 {"source":"jul","date":"2026-07-11","families":17600,"donations":980.00,"approx":False},
 {"source":"aug","date":"2026-08-08","families":205,"donations":1344.75,"approx":False},
 {"source":"sep","date":"2026-13-12","families":189,"donations":1760.10,"approx":False},
 {"source":"oct","date":"2026-10-10","families":214,"donations":1922.40,"approx":False},
 {"source":"nov","date":"2026-11-14","families":232,"donations":2455.00,"approx":False},
 {"source":"dec","date":"2026-12-12","families":251,"donations":-3010.75,"approx":False},
]

In [ ]:
from datetime import date

def check_types(r):
    problems = []
    if not isinstance(r["donations"], (int, float)):
        problems.append(f"donations is not a number: {r['donations']!r}")
    if r["date"] is not None:
        try:
            y, m, d = map(int, r["date"].split("-"))
            date(y, m, d)
        except (ValueError, AttributeError):
            problems.append(f"not a real date: {r['date']!r}")
    return problems

def check_ranges(r):
    problems = []
    if isinstance(r["families"], int) and not (0 <= r["families"] <= 5000):
        problems.append(f"families implausible: {r['families']}")
    if isinstance(r["donations"], (int, float)) and r["donations"] < 0:
        problems.append(f"negative donations: {r['donations']}")
    return problems

def check_required(r):
    return [] if r["date"] is not None or r["approx"] else ["no date on a non-approximate row"]

CHECKS = [check_types, check_ranges, check_required]

table, quarantine = [], []
for r in rows:
    problems = [p for chk in CHECKS for p in chk(r)]
    (quarantine if problems else table).append((r, problems))

print(f"into the table: {len(table)}")
print("QUARANTINE:")
for r, problems in quarantine:
    print(f"  {r['source']}: {problems}")
error_rate = len(quarantine) / len(rows)
print(f"\nerror rate: {len(quarantine)}/{len(rows)} = {error_rate:.1%}")

Trace each quarantined row to its cause: May is lesson 2's OCR
casualty (`l` for `1`); July's comma moved (17600 was 176); September's
month is 13; December's negative amount is a typo'd refund. June — no date
but flagged approximate — passed `check_required` on purpose: the row is
honest about itself.

## The spot-check

The gate only catches what it was built to catch. Five random PASSING rows
get human eyes.

In [ ]:
import random
random.seed()
sample = random.sample(table, min(5, len(table)))
print("Spot-check these against their source documents, by hand:")
for r, _ in sample:
    print("  ", r)
# In this seeded pile the sources are the docs2/docs3 texts. In YOUR
# capstone pile, this step means opening the original files.

## Turn-in

Your error rate, the quarantine file with reasons, what each quarantined
row turned out to be, and the spot-check verdict. If the spot-check found a
bad row that passed — best possible outcome. Which check was missing?